# Sentinel-2 seasonal composites via Google Earth Engine — parallel version

Downloads composites per year for chosen areas, using
**Sentinel-2 SR Harmonized**.

**TODO for LAURA**
1. Create a EART ENGINE project: https://console.cloud.google.com/projectcreate
2. Register the project: https://code.earthengine.google.com/register
3. Run the [authenticate cell](#Earth-Engine-Config) ONCE
4. Inititalize the project, remember to replace with your project name

## 1. Setup

In [ ]:
import os
import re
import time
import logging
from io import BytesIO
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from dateutil.relativedelta import relativedelta


import ee
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from PIL import Image
from shapely import wkt

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

### Earth Engine Config

In [ ]:
# ONLY NEEDED ONCE per machine.
# ee.Authenticate()

In [ ]:
# CREATE A PROJECT IN THE EARTH ENGINE CONSOLE AND USE IT HERE, YOU NEED
# TO THEN REPLACE THE PROJECT NAME BELOW WITH YOUR OWN
ee.Initialize(project="landsat9-502716")

### Config

In [ ]:
OUTPUT_DIR = "output"
NUTS_GEOJSON = "NUTS_RG_20M_2024_4326.geojson"
nuts_gdf = gpd.read_file(NUTS_GEOJSON)

DEFAULT_DIMENSIONS = 1000      # max px on the long side; 1200 is ~40% faster
MAX_WORKERS = 6                # parallel EE requests; 4-8 is safe, higher risks 429s

RGB_BANDS = ["B4", "B3", "B2"]

## 2. Core Earth Engine functions

In [5]:
def mask_clouds(image):
    """Sentinel-2: use the SCL scene classification band.
    3=cloud shadow, 8=cloud medium prob, 9=cloud high prob, 10=cirrus."""
    scl = image.select("SCL")
    clear = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(clear).copyProperties(image, image.propertyNames())


def apply_scale_factors(image, rgb_bands):
    """S2 SR: reflectance = DN / 10000 (no offset)."""
    optical = image.select(rgb_bands).divide(10000)
    return image.addBands(optical, None, True)


def safe_name(name):
    """Strip filesystem-hostile chars: slashes for Cataluña/Catalunya etc."""
    return re.sub(r'[/\\:*?"<>|]', " - ", name)

In [6]:
def get_composite_url(geometry, start_date, end_date,
                      dimensions=DEFAULT_DIMENSIONS, mask_water=False):
    """Build the composite and return a thumbnail URL. No .getInfo() round-trip —
    if the collection is empty, getThumbURL fails and we return None."""
    dataset = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
               .filterDate(start_date.strftime("%Y-%m-%d"),
                           end_date.strftime("%Y-%m-%d"))
               .filterBounds(geometry))

    dataset = dataset.map(mask_clouds)
    dataset = dataset.map(lambda img: apply_scale_factors(img, RGB_BANDS))
    im = dataset.median()

    if mask_water:
        datamask = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").select("datamask")
        im = im.updateMask(datamask.eq(1))

    im = im.updateMask(ee.Image.constant(1).clip(geometry).mask())

    try:
        return im.getThumbURL({
            "bands": RGB_BANDS,
            "min": 0.0,
            "max": 0.35,
            "region": geometry,
            "dimensions": dimensions,
            "format": "png",
            "crs": "EPSG:3857",
        })
    except ee.EEException as e:
        logging.error("Thumbnail URL failed: %s", e)
        return None

## 3. Download functions

The old per-region loop is now split into two things: a `build_jobs()` that lists every
(region, year, window) job, and a `run_jobs()` that runs them in a thread pool.

In [ ]:
def download_image(url, subfolder, fname, timeout=300, retries=3):
    if url is None:
        return
    folder = os.path.join(OUTPUT_DIR, subfolder)
    os.makedirs(folder, exist_ok=True)

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, timeout=timeout)
            if response.status_code != 200:
                logging.error("HTTP %s for %s: %s",
                              response.status_code, fname, response.text[:200])
                return
            Image.open(BytesIO(response.content)).save(
                os.path.join(folder, f"{fname}.png"))
            return
        except requests.exceptions.Timeout:
            logging.warning("Timeout for %s (attempt %s/%s)", fname, attempt, retries)
            time.sleep(5)
    logging.error("Gave up on %s after %s timeouts", fname, retries)


def process_job(job):
    pname, geometry, year, month, mask_water, dimensions = job

    folder_name = safe_name(pname)
    subfolder = os.path.join(folder_name, str(year))
    fname = f"{folder_name}_{year}-{month:02d}-01"

    if os.path.exists(os.path.join(OUTPUT_DIR, subfolder, f"{fname}.png")):
        return fname, 0, "skip"

    t0 = time.time()
    start_date = datetime(year, month, 1)
    end_date = start_date + relativedelta(months=1)

    url = get_composite_url(geometry, start_date, end_date,
                            dimensions=dimensions, mask_water=mask_water)
    t_url = time.time()

    if url is None:
        return fname, t_url - t0, "no-url"

    download_image(url, subfolder, fname)
    t_end = time.time()
    return fname, t_end - t0, f"ok (EE {t_url-t0:.0f}s, dl {t_end-t_url:.0f}s)"

In [ ]:
def build_jobs(regions, start_year, end_year, months=None,
               mask_water=True, dimensions=DEFAULT_DIMENSIONS):
    """regions: list of (pname, geometry) pairs.
    months: list of month numbers (1-12). None or [] means all 12."""
    if not months:
        months = list(range(1, 13))

    jobs = []
    for pname, geometry in regions:
        for year in range(start_year, end_year + 1):
            for month in months:
                jobs.append((pname, geometry, year, month,
                             mask_water, dimensions))
    return jobs


def run_jobs(jobs, max_workers=MAX_WORKERS):
    """Run jobs in parallel. Prints progress and per-job timing."""
    total = len(jobs)
    logging.info("Running %s jobs with %s workers", total, max_workers)
    done = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(process_job, job) for job in jobs]
        for fut in as_completed(futures):
            done += 1
            try:
                fname, elapsed, status = fut.result()
                logging.info("[%d/%d] %s — %s", done, total, fname, status)
            except Exception as e:
                logging.error("[%d/%d] job crashed: %s", done, total, e)

In [ ]:
EUROPE_BBOX = [-13, 34, 45, 72]   # [west, south, east, north]

def nuts_regions(level, countries=None, name_col="NUTS_NAME", id_col="NUTS_ID",
                 clip_bbox=EUROPE_BBOX):
    sel = nuts_gdf[nuts_gdf["LEVL_CODE"] == level]
    if countries is not None:
        sel = sel[sel["CNTR_CODE"].isin(countries)]

    clip = ee.Geometry.Rectangle(clip_bbox) if clip_bbox else None

    out = []
    for _, row in sel.iterrows():
        pname = f"{row[id_col]}_{row[name_col]}"
        geometry = ee.Geometry(row.geometry.__geo_interface__)
        if clip is not None:
            geometry = geometry.intersection(clip, ee.ErrorMargin(1))
        out.append((pname, geometry))
    return out

## 4. Usage examples

Use `build_jobs(regions, start_year, end_year, months)` to get satelitte images. If you leave months empty then every month of the year will be downloaded

In [10]:
nuts_gdf_countries = nuts_gdf[nuts_gdf["LEVL_CODE"] == 0][["CNTR_CODE", "NUTS_NAME"]].drop_duplicates()
all_countries = list(nuts_gdf_countries.CNTR_CODE)

drought_countries = ["ES", "FR", "DE", "IT", "PT", "HU", "PL", "CZ", "LU", "BE", "CH", "NL", "AT"]

In [11]:
regions = nuts_regions(level=0, countries=drought_countries)
jobs = build_jobs(regions, start_year=2020, end_year=2026)
run_jobs(jobs)

INFO: Running 1001 jobs with 6 workers
INFO: [1/1001] AT_Österreich_2020-01-01 — ok (EE 1s, dl 8s)
INFO: [2/1001] AT_Österreich_2020-07-01 — ok (EE 1s, dl 8s)
INFO: [3/1001] AT_Österreich_2020-06-01 — ok (EE 1s, dl 9s)
INFO: [4/1001] AT_Österreich_2020-02-01 — ok (EE 1s, dl 9s)
INFO: [5/1001] AT_Österreich_2020-04-01 — ok (EE 1s, dl 17s)
ERROR: [6/1001] job crashed: month must be in 1..12
INFO: [7/1001] AT_Österreich_2020-03-01 — ok (EE 1s, dl 18s)
INFO: [8/1001] AT_Österreich_2020-09-01 — ok (EE 1s, dl 10s)
INFO: [9/1001] AT_Österreich_2020-08-01 — ok (EE 1s, dl 17s)
INFO: [10/1001] AT_Österreich_2021-07-01 — ok (EE 1s, dl 9s)
INFO: [11/1001] AT_Österreich_2020-10-01 — ok (EE 2s, dl 18s)
INFO: [12/1001] AT_Österreich_2020-11-01 — ok (EE 1s, dl 20s)
INFO: [13/1001] AT_Österreich_2021-01-01 — ok (EE 0s, dl 13s)
INFO: [14/1001] AT_Österreich_2021-02-01 — ok (EE 1s, dl 17s)
INFO: [15/1001] AT_Österreich_2021-06-01 — ok (EE 1s, dl 8s)
INFO: [16/1001] AT_Österreich_2021-08-01 — ok (EE 2s, d

: 

: 